<a href="https://www.kaggle.com/code/stutisharma22/getting-started-titanic-ml-from-disaster?scriptVersionId=268729872" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


# Importing necessary libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.ensemble import VotingClassifier

# Loading the dataset

In [3]:
train= pd.read_csv('/kaggle/input/titanic/train.csv')
test= pd.read_csv('/kaggle/input/titanic/test.csv')

In [4]:
train['Dataset'] = 'train'
test['Dataset'] = 'test'

In [5]:
combined= pd.concat ([train,test], ignore_index= True)

# Viewing first few rows

In [6]:
combined.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Dataset
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,train
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,train
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,train
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,train
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,train


# Exploratory data analysis

In [7]:
combined.shape

(1309, 13)

In [8]:
combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
 12  Dataset      1309 non-null   object 
dtypes: float64(3), int64(4), object(6)
memory usage: 133.1+ KB


In [9]:
combined.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,1309.000000,891.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000
mean,655.000000,0.383838,2.294882,29.881138,0.498854,0.385027,33.295479
std,378.020061,0.486592,0.837836,14.413493,1.041658,0.865560,51.758668
min,1.000000,0.000000,1.000000,0.170000,0.000000,0.000000,0.000000
25%,328.000000,0.000000,2.000000,21.000000,0.000000,0.000000,7.895800
50%,655.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,982.000000,1.000000,3.000000,39.000000,1.000000,0.000000,31.275000
max,1309.000000,1.000000,3.000000,80.000000,8.000000,9.000000,512.329200


In [10]:
combined.nunique()

PassengerId    1309
Survived          2
Pclass            3
Name           1307
Sex               2
Age              98
SibSp             7
Parch             8
Ticket          929
Fare            281
Cabin           186
Embarked          3
Dataset           2
dtype: int64

In [11]:
combined.corr(numeric_only= True)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.000000,-0.005007,-0.038354,0.028814,-0.055224,0.008942,0.031428
Survived,-0.005007,1.000000,-0.338481,-0.077221,-0.035322,0.081629,0.257307
Pclass,-0.038354,-0.338481,1.000000,-0.408106,0.060832,0.018322,-0.558629
Age,0.028814,-0.077221,-0.408106,1.000000,-0.243699,-0.150917,0.178740
SibSp,-0.055224,-0.035322,0.060832,-0.243699,1.000000,0.373587,0.160238
Parch,0.008942,0.081629,0.018322,-0.150917,0.373587,1.000000,0.221539
Fare,0.031428,0.257307,-0.558629,0.178740,0.160238,0.221539,1.000000


# Cleaning the data

In [12]:
combined.isna().sum()

PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
Dataset           0
dtype: int64

In [13]:
combined['Age'] = combined['Age'].fillna(combined.groupby(['Pclass', 'Sex'])['Age'].transform('median'))

In [14]:
combined['Embarked'] = combined['Embarked'].fillna(combined['Embarked'].mode()[0])

In [15]:
combined['Fare'] = combined['Fare'].fillna(combined.groupby('Pclass')['Fare'].transform('median'))

In [16]:
combined['Sex']= combined['Sex'].map({'male':1 , 'female': 0})

In [17]:
combined['FamilySize'] = combined['SibSp'] + combined['Parch'] + 1

In [18]:
combined['Cabin']= combined['Cabin'].fillna('U')
combined['Cabin']= combined['Cabin'].map(lambda x: x[0])

# Feature Engineering

In [19]:
def extract_ticket_prefix(ticket):
    # Remove dots and slashes for consistency
    ticket = ticket.replace('.', '').replace('/', '').strip()
    
    # Split by space
    parts = ticket.split()
    
    # If it has a non-numeric prefix, return it
    if len(parts) > 1 and not parts[0].isdigit():
        return parts[0].upper()
    else:
        return 'NONE'  # for purely numeric tickets

combined['TicketPrefix'] = combined['Ticket'].apply(extract_ticket_prefix)

In [20]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
combined['TicketPrefix'] = le.fit_transform(combined['TicketPrefix'])

In [21]:
combined['FarePerPerson'] = combined['Fare']/ combined['FamilySize']

In [22]:
combined['Title']= combined['Name'].str.extract(" ([A-Za-z]+)\. ", expand= False)
combined['Title'].value_counts()

Title
Mr          757
Miss        260
Mrs         197
Master       61
Rev           8
Dr            8
Col           4
Mlle          2
Major         2
Ms            2
Lady          1
Sir           1
Mme           1
Don           1
Capt          1
Countess      1
Jonkheer      1
Dona          1
Name: count, dtype: int64

In [23]:
combined['Title'] = combined['Title'].replace({
    'Mlle': 'Miss',
    'Ms': 'Miss',
    'Mme': 'Mrs',
    'Countess': 'Royalty',
    'Lady': 'Royalty',
    'Sir': 'Royalty',
    'Don': 'Royalty',
    'Dona': 'Royalty',
    'Jonkheer': 'Royalty',
    'Dr': 'Officer',
    'Major': 'Officer',
    'Col': 'Officer',
    'Capt': 'Officer',
    'Rev': 'Officer'
})

In [24]:
encoder = LabelEncoder()
combined['Title'] = encoder.fit_transform(combined['Title'])
combined['Cabin'] = encoder.fit_transform(combined['Cabin'])
combined['Embarked'] = encoder.fit_transform(combined['Embarked'])


In [25]:
combined['isAlone'] = (combined['FamilySize']== 1).astype(int)

In [26]:
combined['AgeClass'] = combined['Age'] * combined['Pclass']

In [27]:
combined.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Dataset,FamilySize,TicketPrefix,FarePerPerson,Title,isAlone,AgeClass
0,1,0.0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,A/5 21171,7.2500,8,2,train,2,2,3.62500,2,0,66.0
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,PC 17599,71.2833,2,0,train,2,14,35.64165,3,0,38.0
2,3,1.0,3,"Heikkinen, Miss. Laina",0,26.0,0,0,STON/O2. 3101282,7.9250,8,2,train,1,30,7.92500,1,1,78.0
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,113803,53.1000,2,2,train,2,13,26.55000,3,0,35.0
4,5,0.0,3,"Allen, Mr. William Henry",1,35.0,0,0,373450,8.0500,8,2,train,1,13,8.05000,2,1,105.0


In [28]:
train_cleaned= combined[combined['Dataset'] == 'train'].drop(columns=['Dataset', 'Name', 'Ticket'])
test_cleaned= combined[combined['Dataset'] == 'test'].drop(columns=['Dataset','Name', 'Ticket','Survived'])

In [29]:
test_cleaned.head()

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked,FamilySize,TicketPrefix,FarePerPerson,Title,isAlone,AgeClass
891,892,3,1,34.5,0,0,7.8292,8,1,1,13,7.829200,2,1,103.5
892,893,3,0,47.0,1,0,7.0000,8,2,2,13,3.500000,3,0,141.0
893,894,2,1,62.0,0,0,9.6875,8,1,1,13,9.687500,2,1,124.0
894,895,3,1,27.0,0,0,8.6625,8,2,1,13,8.662500,2,1,81.0
895,896,3,0,22.0,1,1,12.2875,8,2,3,13,4.095833,3,0,66.0


In [30]:
X= train_cleaned.drop('Survived', axis=1)
y= train_cleaned['Survived']

In [31]:
X_train,X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state= 42, stratify= y)

# Training the model

In [32]:
model1 = RandomForestClassifier(n_estimators=300, random_state=42)
model2= XGBClassifier(n_estimators= 500, learning_rate=0.02)
model3= lgb.LGBMClassifier(num_leaves= 8, learning_rate=0.03)

In [33]:
voting_clf= VotingClassifier(
    estimators= [
        ('rf', model1), ('xgb', model2), ('lgb', model3)
    ], voting= 'soft'
)

In [34]:
voting_clf.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002789 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 690
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028


VotingClassifier(estimators=[('rf',
                              RandomForestClassifier(n_estimators=300,
                                                     random_state=42)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            grow_policy=No...
                                            learning_rate=0.02, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=500, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=None, ...)),
                             ('lgb',
                              LGBMClassifier(learning_rate=0.03,
                                             num_leaves=8))],
                 voting='soft')

In [35]:
y_pred= voting_clf.predict(X_test)

In [36]:
accuracy= accuracy_score(y_test, y_pred)
print(accuracy)

0.7932960893854749


# Output

In [37]:
output= pd.DataFrame({'PassengerId':test.PassengerId, 'Survived': voting_clf.predict(test_cleaned).astype(int)})
output.to_csv('submission.csv', index= False)